# Create synthetic spectra with `minato.synthetic`

This notebook creates synthetic single-star and binary spectra that can be passed to RAVEL. It replaces the older notebook-local renderer with the package API in `minato.synthetic`.

The example is runnable without external data because it uses a small analytic atmosphere backend. For science use, replace that backend with `TextAtmosphereGrid.from_directory(...)` pointing at your own atmosphere-model files.

## What this tutorial does

- Define a simple atmosphere backend.
- Show how to switch to user-supplied PoWR/TLUSTY/FASTWIND-style text grids.
- Render one single-star example.
- Render multi-epoch SB1 and SB2 examples.
- Write RAVEL-compatible text spectra plus `JDs.txt` and a truth manifest.

Outputs are written to `tutorial_outputs/create_synth_spectra/` so this notebook does not overwrite the committed RAVEL tutorial data.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from minato.synthetic import (
    BinarySystem,
    FallbackAtmosphereGrid,
    ObservationModel,
    Spectrum,
    Star,
    TextAtmosphereGrid,
    render_binary,
    render_single_star,
)
from minato.synthetic.io import write_ravel_txt

## Atmosphere model backend

`render_single_star` and `render_binary` need an atmosphere-grid backend with a `get_spectrum(star)` method. The backend returns either a `Spectrum` or a `(wavelength, flux)` tuple.

The analytic backend below is pedagogical. It makes line-like absorption features near common RAVEL lines so the rest of the workflow can run without downloading or bundling large atmosphere grids.

In [ ]:
class AnalyticOBGrid:
    """Tiny atmosphere backend for tutorial use only."""

    def __init__(self, wavelength_min=3900.0, wavelength_max=7000.0, n_pixels=12000):
        self.wavelength = np.linspace(wavelength_min, wavelength_max, n_pixels)
        self.lines = np.array([4026.0, 4089.0, 4102.0, 4144.0, 4340.0, 4388.0, 4471.0, 4542.0, 4553.0])

    def get_spectrum(self, star):
        flux = np.ones_like(self.wavelength)
        teff_scale = np.clip((36_000.0 - star.teff) / 24_000.0, 0.15, 0.95)
        logg_scale = np.clip((4.7 - star.logg) / 1.2, 0.25, 1.0)
        rotation_scale = 1.0 + 0.0015 * star.vsini

        for index, centre in enumerate(self.lines):
            depth = (0.05 + 0.015 * index) * teff_scale * logg_scale
            width = (0.45 + 0.05 * index) * rotation_scale
            flux -= depth * np.exp(-0.5 * ((self.wavelength - centre) / width) ** 2)

        return Spectrum(
            self.wavelength,
            flux,
            metadata={"atmosphere_backend": "analytic_ob_grid"},
        )


grid = AnalyticOBGrid()

## Using real model grids

For real work, users should provide their own atmosphere model files. MINATO can scan directories of text spectra and infer `teff`/`logg` from recognised MINATO, PoWR, TLUSTY, or FASTWIND-style names.

If a preferred grid has partial coverage, use `FallbackAtmosphereGrid` to try grids in priority order. Each grid decides whether its nearest node is acceptable using the tolerances you set.

In [ ]:
USE_REAL_MODEL_GRID = False

if USE_REAL_MODEL_GRID:
    powr_grid = TextAtmosphereGrid.from_directory(
        "path/to/powr_models",
        format="powr",
        max_teff_delta=800,
        max_logg_delta=0.25,
    )
    tlusty_grid = TextAtmosphereGrid.from_directory(
        "path/to/tlusty_models",
        format="tlusty",
        max_teff_delta=1000,
        max_logg_delta=0.25,
    )
    grid = FallbackAtmosphereGrid([
        ("powr", powr_grid),
        ("tlusty", tlusty_grid),
    ])

For unconventional local filenames, either pass `filename_pattern=...`, pass a parser function, create an editable CSV with `TextAtmosphereGrid.write_index_template(...)`, or symlink/rename files to the MINATO convention, for example `teff25000_logg4.00.txt`.

## Observation model

`ObservationModel` controls the output wavelength range, resolving power, velocity-grid spacing, S/N, and random seed. Use a different seed per epoch when creating a time series.

In [ ]:
base_observation = ObservationModel(
    resolving_power=4000,
    snr=50,
    wavelength_min=3900.0,
    wavelength_max=7000.0,
    velocity_step=5.0,
    seed=42,
)

output_root = Path("tutorial_outputs/create_synth_spectra")
output_root

## Render one single-star spectrum

In [ ]:
single_star = Star(teff=25_000, logg=4.0, radius=8.0, rv=120.0, vsini=80.0, label="single")
single_spectrum = render_single_star(single_star, atmosphere_grid=grid, observation=base_observation)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(single_spectrum.wavelength, single_spectrum.flux, lw=1)
ax.set_xlim(4000, 4600)
ax.set_xlabel("Wavelength (Angstrom)")
ax.set_ylabel("Flux")
plt.show()

## Define a deterministic cadence and RV curves

The RV curves here are simple sinusoids for a compact tutorial. For science simulations, use your own orbital solution or a population simulator, then pass the epoch RVs into `Star`.

In [ ]:
n_epochs = 12
epoch = np.arange(1, n_epochs + 1)
mjd = 60_000.0 + np.array([0.0, 1.4, 3.1, 4.6, 6.2, 8.0, 10.3, 13.2, 17.4, 22.0, 27.5, 31.4])
phase = 2.0 * np.pi * (mjd - mjd[0]) / 12.0

epochs = pd.DataFrame(
    {
        "epoch": [f"epoch_{i:02d}" for i in epoch],
        "mjd": mjd,
        "rv_sb1": 35.0 * np.sin(phase) + 10.0,
        "rv1_sb2": 75.0 * np.sin(phase) + 5.0,
        "rv2_sb2": -125.0 * np.sin(phase) + 5.0,
    }
)
epochs.head()

## Helpers for writing RAVEL inputs

In [ ]:
def observation_for_epoch(base, epoch_index):
    return ObservationModel(
        resolving_power=base.resolving_power,
        snr=base.snr,
        wavelength_min=base.wavelength_min,
        wavelength_max=base.wavelength_max,
        velocity_step=base.velocity_step,
        limb_darkening=base.limb_darkening,
        seed=None if base.seed is None else base.seed + int(epoch_index),
    )


def write_jds(table, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    table[["epoch", "mjd"]].to_csv(output_dir / "JDs.txt", sep=" ", header=False, index=False)


def write_manifest(rows, output_dir):
    manifest = pd.DataFrame(rows)
    manifest.to_csv(output_dir / "truth_manifest.csv", index=False)
    return manifest

## Create an SB1 example

This creates one visible spectrum per epoch. The companion's effect is represented only through the primary-star RV curve.

In [ ]:
sb1_dir = output_root / "SB1"
sb1_rows = []

for index, row in epochs.iterrows():
    star = Star(
        teff=25_000,
        logg=4.0,
        radius=8.0,
        rv=row.rv_sb1,
        vsini=80.0,
        label="sb1_primary",
    )
    spectrum = render_single_star(
        star,
        atmosphere_grid=grid,
        observation=observation_for_epoch(base_observation, index),
    )
    filename = sb1_dir / f"synthetic_SB1_{index + 1:02d}.txt"
    write_ravel_txt(spectrum, filename)
    sb1_rows.append({"epoch": row.epoch, "mjd": row.mjd, "filename": filename.name, "rv_primary": row.rv_sb1})

write_jds(epochs, sb1_dir)
sb1_manifest = write_manifest(sb1_rows, sb1_dir)
sb1_manifest.head()

## Create an SB2 example

This combines a primary and secondary spectrum in memory, applies their epoch RVs, adds reproducible noise, and writes one RAVEL text file per epoch.

In [ ]:
sb2_dir = output_root / "SB2"
sb2_rows = []

for index, row in epochs.iterrows():
    system = BinarySystem(
        primary=Star(teff=30_000, logg=4.1, radius=9.0, rv=row.rv1_sb2, vsini=100.0, label="primary"),
        secondary=Star(teff=18_000, logg=4.2, radius=4.5, rv=row.rv2_sb2, vsini=70.0, label="secondary"),
    )
    spectrum = render_binary(
        system,
        atmosphere_grid=grid,
        observation=observation_for_epoch(base_observation, 100 + index),
    )
    filename = sb2_dir / f"synthetic_SB2_{index + 1:02d}.txt"
    write_ravel_txt(spectrum, filename)
    sb2_rows.append(
        {
            "epoch": row.epoch,
            "mjd": row.mjd,
            "filename": filename.name,
            "rv_primary": row.rv1_sb2,
            "rv_secondary": row.rv2_sb2,
            "primary_weight": spectrum.metadata["primary_weight"],
            "secondary_weight": spectrum.metadata["secondary_weight"],
        }
    )

write_jds(epochs, sb2_dir)
sb2_manifest = write_manifest(sb2_rows, sb2_dir)
sb2_manifest.head()

## Inspect the generated spectra

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for filename in sorted(sb2_dir.glob("synthetic_SB2_*.txt"))[:4]:
    data = np.loadtxt(filename)
    ax.plot(data[:, 0], data[:, 1], lw=0.8, label=filename.stem)

ax.set_xlim(4000, 4600)
ax.set_xlabel("Wavelength (Angstrom)")
ax.set_ylabel("Flux")
ax.legend(ncol=2, fontsize=8)
plt.show()

## Passing the files to RAVEL

The generated text files have the same shape accepted by `ravel.read_spectra`. Importing `minato.ravel` initialises the JAX/NumPyro stack, so keep this optional in a generation notebook. Run the following only when you are ready to inspect or fit the spectra.

In [ ]:
RUN_RAVEL_READ = False

if RUN_RAVEL_READ:
    from minato import ravel

    spec_files = sorted(str(path) for path in sb2_dir.glob("synthetic_SB2_*.txt"))
    wavelengths, fluxes, f_errors, names, jds = ravel.read_spectra(
        spec_files,
        path=str(sb2_dir),
        file_type="txt",
    )
    print(len(wavelengths), names[:2], jds[:2])